# AdaBoost hyperparameters
`AdaBoostClassifier` on noisy concentric circles: default settings, the effect of `n_estimators` and
`learning_rate`, which base models are allowed, and a grid search. The decision-surface demo is `app.py`
(`python app.py`, then open http://127.0.0.1:8050); the cells below use the same code.

In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from app import X_test, X_train, figure, fit, y_test, y_train

## 1. The data and a default AdaBoost

In [ ]:
# 500 points: a small blue disc inside an orange ring, with heavy noise
X, y = make_circles(n_samples=500, factor=0.1, noise=0.35, random_state=42)

# default: 50 stumps (trees with max_depth=1), learning_rate=1.0
abc = AdaBoostClassifier(random_state=42)
print("10-fold CV accuracy:", cross_val_score(abc, X, y, cv=10, scoring="accuracy").mean())
abc.fit(X, y)
print(abc.estimator_)   # the default base model

## 2. n_estimators: from underfitting to overfitting

In [ ]:
panels = []
for n in (1, 50, 150, 500, 1500):
    model, acc = fit(n_estimators=n, learning_rate=1.0)
    panels.append((f"n_estimators={n}: {acc}", model))
    print(panels[-1][0])
figure(panels)

## 3. learning_rate: shrinking every alpha

In [ ]:
model, acc = fit(n_estimators=1500, learning_rate=0.1)
print("1500 stumps, learning_rate=0.1:", acc)
figure([("learning_rate=1.0: " + panels[-1][0].split(": ")[1], panels[-1][1]),
        (f"learning_rate=0.1: {acc}", model)])

In [ ]:
# alpha with learning_rate 1.0 and 0.1: the first stump's weight is 10 times smaller
for lr in (1.0, 0.1):
    m = AdaBoostClassifier(n_estimators=3, learning_rate=lr, random_state=42).fit(X_train, y_train)
    print(lr, m.estimator_weights_.round(4))

In [ ]:
# accuracy after each added stump, for three learning rates
for lr in (1.0, 0.1, 0.01):
    m = AdaBoostClassifier(n_estimators=1500, learning_rate=lr, random_state=42).fit(X_train, y_train)
    test = np.array(list(m.staged_score(X_test, y_test)))
    print(f"learning_rate={lr}: best test accuracy {test.max():.2f} after {test.argmax() + 1} stumps,"
          f" {test[-1]:.2f} after 1500")

## 4. The base model must accept sample weights

In [ ]:
# KNN has no sample_weight in fit, so AdaBoost refuses it
try:
    AdaBoostClassifier(KNeighborsClassifier()).fit(X, y)
except ValueError as e:
    print("ValueError:", e)

## 5. Grid search

In [ ]:
grid = {
    "n_estimators": [10, 50, 100, 500],
    "learning_rate": [0.0001, 0.001, 0.01, 0.1, 1.0],
}
grid_search = GridSearchCV(AdaBoostClassifier(random_state=42), param_grid=grid,
                           n_jobs=-1, cv=10, scoring="accuracy")
grid_result = grid_search.fit(X, y)
print(f"Best: {grid_result.best_score_:.3f} using {grid_result.best_params_}")

## 6. Extra: deeper base trees

In [ ]:
grid2 = {
    "estimator": [DecisionTreeClassifier(max_depth=d) for d in (1, 2, 3)],
    "n_estimators": [10, 50, 100, 500],
    "learning_rate": [0.01, 0.1, 1.0],
}
gs2 = GridSearchCV(AdaBoostClassifier(random_state=42), grid2, n_jobs=-1, cv=10).fit(X, y)
print(f"Best: {gs2.best_score_:.3f} using {gs2.best_params_}")

for d in (1, 3, 8):
    model, acc = fit(n_estimators=50, learning_rate=1.0, max_depth=d)
    print(f"max_depth={d}, 50 trees: {acc}")